# private-targeting Colab demo

This notebook is a fast smoke-test tutorial for the `private-targeting` package accompanying the paper:

> Ponte, Gilian R., Tom Boot, Thomas Reutterer, and Jaap E. Wieringa. “EXPRESS: Where Should Firms Implement Differential Privacy in Targeting? Implications for Profitability.” *Journal of Marketing Research*, 2026. DOI: `10.1177/00222437261455302`.

The package exposes three public functions:

- `CTENN`: non-private CATE estimator.
- `DP_CATE`: differentially private CATE estimator.
- `DP_policy`: randomized-response targeting policy evaluation.

This notebook uses a tiny synthetic dataset and very small training settings so that it runs quickly in Colab. It is meant as a smoke test, not as a replication of the paper's empirical results.

## 1. Check Python version

The package is intended for Python **3.10 or 3.11** because the full ML stack depends on `tensorflow-privacy==0.9.0`, which requires Python `<3.12`.

In [ ]:
import sys
print(sys.version)

## 2. Install the package

Run the cell below once. After installation, restart the Colab runtime using:

**Runtime → Restart session**

Then continue from the import test cell below.

In [ ]:
%pip install --upgrade pip setuptools wheel
%pip install "private-targeting[full] @ git+https://github.com/GilianPonte/private-targeting.git@v0.1.0"

## 3. Import test

After restarting the runtime, run this cell to check that the package imports correctly.

In [ ]:
from private_targeting import CTENN, DP_CATE, DP_policy

print("Import works")

## 4. Create a toy targeting dataset

The toy dataset has:

- customer features `X`;
- binary treatment assignment `T`;
- a known true heterogeneous treatment effect `true_cate`;
- an observed outcome `Y`.

In [ ]:
import os

# Keep TensorFlow logs quieter in small tutorial runs.
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(1)

n = 100
p = 5

X = rng.normal(size=(n, p))
T = rng.binomial(1, 0.5, size=n)

true_cate = 1 + 0.5 * X[:, 0] - 0.25 * X[:, 1]
Y = 2 + X[:, 0] + X[:, 1] + T * true_cate + rng.normal(scale=1, size=n)

print("X shape:", X.shape)
print("Y shape:", Y.shape)
print("T shape:", T.shape)
print("Mean true CATE:", round(float(np.mean(true_cate)), 4))

## 5. Estimate CATEs without privacy protection: `CTENN`

`CTENN` estimates individual-level conditional average treatment effects without privacy protection.

It returns:

- `ate_ctenn`: estimated average treatment effect;
- `cate_ctenn`: individual-level CATE estimates;
- `model_ctenn`: fitted neural-network model.

In [ ]:
ate_ctenn, cate_ctenn, model_ctenn = CTENN(
    X=X,
    Y=Y,
    T=T,
    folds=2,
    epochs=1,
    max_epochs=1,
    batch_size=10,
    directory="colab_tuner/ctenn",
    seed=1,
)

print("CTENN ATE:", ate_ctenn)
print("First five CTENN CATE estimates:", cate_ctenn[:5])

## 6. Estimate CATEs with differential privacy: `DP_CATE`

`DP_CATE` estimates individual-level conditional average treatment effects with differential privacy during model training.

It returns:

- `ate_dp`: differentially private estimated average treatment effect;
- `cate_dp`: differentially private individual-level CATE estimates;
- `model_dp`: fitted private neural-network model;
- `n_dp`: number of observations used for privacy accounting;
- `epsilon`: estimated privacy-loss parameter;
- `noise`: noise multiplier used during private optimization;
- `epsilon_conservative`: conservative privacy-loss estimate.

The tutorial uses `fixed_model=True`, `epochs=1`, and `max_epochs=1` to keep the run fast.

In [ ]:
ate_dp, cate_dp, model_dp, n_dp, epsilon, noise, epsilon_conservative = DP_CATE(
    X=X,
    Y=Y,
    T=T,
    epochs=1,
    max_epochs=1,
    batch_size=10,
    directory="colab_tuner/dp_cate",
    noise_multiplier=1.0,
    fixed_model=True,
    seed=1,
)

print("DP_CATE ATE:", ate_dp)
print("n used for privacy accounting:", n_dp)
print("noise multiplier:", noise)
print("epsilon:", epsilon)
print("epsilon conservative:", epsilon_conservative)
print("First five DP_CATE estimates:", cate_dp[:5])

## 7. Evaluate protected targeting policies: `DP_policy`

`DP_policy` evaluates targeting decisions under randomized-response privacy protection.

It compares:

- `real`: oracle targeting using true CATEs;
- `CTENN`: targeting using non-private CTENN estimates;
- one row for each privacy level in `epsilons`;
- `random`: random targeting, used as the baseline.

The output is a pandas DataFrame with `percent`, `epsilon`, `difference_from_random`, and `iteration`.

In [ ]:
policy_results = DP_policy(
    iterations=2,
    percentage=[0.10, 0.20],
    CATE=true_cate,
    CATE_estimates=cate_ctenn,
    epsilons=[0.5, 1, 3],
    seed_offset=1,
    verbose=False,
)

policy_results

## 8. Plot CTENN CATE estimates

This scatter plot compares the estimated CTENN CATEs with the known true CATEs from the toy data-generating process.

In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(true_cate, cate_ctenn, alpha=0.6)
plt.axline((0, 0), slope=1, linestyle="--")
plt.xlabel("True CATE")
plt.ylabel("Estimated CATE from CTENN")
plt.title("CTENN CATE estimates")
plt.tight_layout()
plt.show()

## 9. Plot DP-policy profit

This plot shows mean profit above random targeting for each policy and targeting fraction.

In [ ]:
summary = (
    policy_results
    .groupby(["percent", "epsilon"], as_index=False)
    .agg(mean_profit=("difference_from_random", "mean"))
)

plt.figure(figsize=(7, 4))

for eps, group in summary.groupby("epsilon"):
    group = group.sort_values("percent")
    plt.plot(group["percent"], group["mean_profit"], marker="o", label=str(eps))

plt.axhline(0, linestyle="--")
plt.xlabel("Targeted fraction")
plt.ylabel("Profit above random policy")
plt.title("DP-policy profit")
plt.legend(title="Policy")
plt.tight_layout()
plt.show()

## 10. Next steps

For real experiments, increase the sample size, training epochs, tuning budget, and number of policy iterations. The settings in this notebook are intentionally small so the example runs quickly in Colab.